In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!sudo apt-get install -y gromacs
# install gromacs in the system 

In [ ]:
!gmx --version
# check the version

In [ ]:
!git clone https://github.com/Alcartez/Gromacs_2024.4_stable_installer.git
%cd Gromacs_2024.4_stable_installer
!bash install_conda_kaggle.sh

In [ ]:
%%bash
# =====================================================================
# 📁 CONFIGURATION: Set your Kaggle input path here
# =====================================================================
INPUT_PDB="/kaggle/input/YOUR_KAGGLE_USERNAME/YOUR_DATASET_NAME/dla_complex.pdb"
CLEAN_PDB="/kaggle/working/dla_clean.pdb"

echo "=== 1. Stripping crystallographic water (HOH) ==="

if [ -f "$INPUT_PDB" ]; then
    grep -v "HOH" "$INPUT_PDB" > "$CLEAN_PDB"
    echo "[SUCCESS] Cleaned PDB generated at: $CLEAN_PDB"
else
    echo "[ERROR] Input PDB file not found! Please check your Kaggle input path."
    exit 1
fi

echo ""
echo "=== 2. Verifying output file creation and size ==="
ls -lh "$CLEAN_PDB"

echo ""
echo "=== 3. Checking for remaining non-protein residues (HETATM) ==="
grep "^HETATM" "$CLEAN_PDB" | awk '{print $4}' | sort -u

In [ ]:
%%bash
echo "6" | gmx pdb2gmx \
  -f /kaggle/working/dla_clean.pdb \
  -o /kaggle/working/dla_processed.gro \
  -p /kaggle/working/topol.top \
  -i /kaggle/working/posre.itp \
  -water tip3p \
  -ignh

In [ ]:
%%bash
echo "=== Checking the contents of /kaggle/working/ ==="
ls -lh /kaggle/working/

In [ ]:
%%bash
cat /kaggle/working/topol.top

In [ ]:
%%bash

echo "=== Running editconf to create the cubic unit cell ==="
gmx editconf \
  -f /kaggle/working/dla_processed.gro \
  -o /kaggle/working/dla_newbox.gro \
  -c \
  -d 1.2 \
  -bt cubic

echo ""
echo "=== Running solvate to add TIP3P water molecules ==="
gmx solvate \
  -cp /kaggle/working/dla_newbox.gro \
  -cs spc216.gro \
  -o /kaggle/working/dla_solv.gro \
  -p /kaggle/working/topol.top

echo ""
echo "=== Verifying updated [ molecules ] section at the bottom of topol.top ==="
tail -n 10 /kaggle/working/topol.top

In [ ]:
# First, create the ions.mdp file in the working directory
mdp_content = """; ions.mdp - used as input into grompp to generate ions.tpr
integrator  = steep         ; Algorithm (steep = steepest descent minimization)
emtol       = 1000.0        ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
emstep      = 0.01          ; Minimization step size
nsteps      = 50000         ; Maximum number of (minimization) steps to perform

; Parameters describing how to find the neighbors of each atom and how to calculate the interactions
nstlist         = 1         ; Frequency to update the neighbor list and long range forces
cutoff-scheme	= Verlet    ; Buffered neighbor searching 
ns_type         = grid      ; Method to determine neighbor list (simple, grid)
coulombtype     = cutoff    ; Treatment of long range electrostatic interactions
rcoulomb        = 1.0       ; Short-range electrostatic cut-off
rvdw            = 1.0       ; Short-range Van der Waals cut-off
pbc             = xyz       ; Periodic Boundary Conditions in all 3 dimensions
"""

with open('/kaggle/working/ions.mdp', 'w') as f:
    f.write(mdp_content)
print("ions.mdp successfully created.")

In [ ]:
%%bash

echo "=== Running grompp to assemble ions.tpr ==="
gmx grompp \
  -f /kaggle/working/ions.mdp \
  -c /kaggle/working/dla_solv.gro \
  -p /kaggle/working/topol.top \
  -o /kaggle/working/ions.tpr

echo ""
echo "=== Running genion to neutralize the system charge ==="
# We pass "SOL" automatically to select the solvent group for ion substitution
echo "SOL" | gmx genion \
  -s /kaggle/working/ions.tpr \
  -o /kaggle/working/dla_solv_ions.gro \
  -p /kaggle/working/topol.top \
  -pname NA \
  -nname CL \
  -neutral

echo ""
echo "=== Verifying the updated [ molecules ] section ==="
tail -n 10 /kaggle/working/topol.top

In [ ]:
# Create the minim.mdp file in the working directory
minim_content = """; minim.mdp - used as input into grompp to generate em.tpr
integrator  = steep         ; Algorithm (steep = steepest descent minimization)
emtol       = 1000.0        ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
emstep      = 0.01          ; Minimization step size
nsteps      = 50000         ; Maximum number of (minimization) steps to perform

; Parameters describing how to find the neighbors of each atom and how to calculate the interactions
nstlist         = 1         ; Frequency to update the neighbor list and long range forces
cutoff-scheme   = Verlet    ; Buffered neighbor searching
ns_type         = grid      ; Method to determine neighbor list (simple, grid)
coulombtype     = PME       ; Treatment of long range electrostatic interactions
rcoulomb        = 1.0       ; Short-range electrostatic cut-off
rvdw            = 1.0       ; Short-range Van der Waals cut-off
pbc             = xyz       ; Periodic Boundary Conditions in all 3 dimensions
"""

with open('/kaggle/working/minim.mdp', 'w') as f:
    f.write(minim_content)
print("minim.mdp successfully created.")

In [ ]:
%%bash

echo "=== Running grompp to assemble em.tpr ==="
gmx grompp \
  -f /kaggle/working/minim.mdp \
  -c /kaggle/working/dla_solv_ions.gro \
  -p /kaggle/working/topol.top \
  -o /kaggle/working/em.tpr

echo ""
echo "=== Running mdrun Energy Minimization (CPU mode) ==="
gmx mdrun \
  -v \
  -deffnm /kaggle/working/em

In [ ]:
# Create the nvt.mdp file tailored for your Amber99sb complex setup
nvt_content = """; nvt.mdp - Amber99sb-ildn optimized NVT equilibration
title                   = Amber Complex NVT equilibration 
define                  = -DPOSRES  ; position restrain the protein chains

; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = 50000     ; 2 * 50000 = 100 ps
dt                      = 0.002     ; 2 fs

; Output control
nstxout                 = 2500      ; save coordinates every 5.0 ps
nstvout                 = 2500      ; save velocities every 5.0 ps
nstenergy               = 2500      ; save energies every 5.0 ps
nstlog                  = 2500      ; update log file every 5.0 ps

; Bond parameters
continuation            = no        ; first dynamics run
constraint_algorithm    = lincs     ; holonomic constraints 
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy

; Nonbonded settings 
cutoff-scheme           = Verlet    ; Buffered neighbor searching
nstlist                 = 10        ; 20 fs neighbor list update

; Electrostatics & VdW (Amber defaults)
rcoulomb                = 1.0       ; short-range electrostatic cutoff (nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (nm)
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT

; Temperature coupling is on
tcoupl                  = V-rescale             ; stochastic Bussi thermostat 
tc-grps                 = Protein Non-Protein   ; Two coupling groups for multi-chain setups
tau_t                   = 0.1     0.1           ; time constant (ps)
ref_t                   = 298     298           ; reference temperature (K) 

; Pressure coupling is off
pcoupl                  = no        ; no pressure coupling in NVT

; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC

; Velocity generation
gen_vel                 = yes       ; assign velocities from Maxwell distribution
gen_temp                = 298       ; temperature for Maxwell distribution
gen_seed                = -1        ; generate a random seed
"""

with open('/kaggle/working/nvt.mdp', 'w') as f:
    f.write(nvt_content)
print("nvt.mdp successfully created.")

In [ ]:
!nvidia-smi

In [ ]:
!micromamba run -n gmx2024_cuda gmx --version

In [ ]:
!micromamba remove -n gmx2024_cuda --all -y

In [ ]:
!micromamba create -n gmx2024_cuda -c conda-forge \
  "gromacs=2026.*=*nompi_cuda*" cuda-version=12 -y

In [ ]:
!micromamba run -n gmx2024_cuda gmx --version

In [ ]:
%%bash

# Set up the shortcut alias for the micromamba GROMACS environment binary
GMX_ENV="micromamba run -n gmx2024_cuda gmx"

echo "=== Running grompp to assemble nvt.tpr ==="
$GMX_ENV grompp \
  -f /kaggle/working/nvt.mdp \
  -c /kaggle/working/em.gro \
  -r /kaggle/working/em.gro \
  -p /kaggle/working/topol.top \
  -o /kaggle/working/nvt.tpr

echo ""
echo "=== Running mdrun NVT Equilibration on GPU ==="
$GMX_ENV mdrun \
  -v \
  -deffnm /kaggle/working/nvt \
  -nb gpu \
  -pme gpu

In [ ]:
# Create the npt.mdp file tailored for your Amber99sb complex setup
npt_content = """; npt.mdp - Amber99sb-ildn optimized NPT equilibration
title                   = Amber Complex NPT equilibration 
define                  = -DPOSRES  ; position restrain the protein chains

; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = 250000    ; 2 * 250000 = 500 ps
dt                      = 0.002     ; 2 fs

; Output control
nstxout                 = 2500      ; save coordinates every 5.0 ps
nstvout                 = 2500      ; save velocities every 5.0 ps
nstenergy               = 2500      ; save energies every 5.0 ps
nstlog                  = 2500      ; update log file every 5.0 ps

; Bond parameters
continuation            = yes       ; Restarting after NVT
constraint_algorithm    = lincs     ; holonomic constraints 
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy

; Nonbonded settings 
cutoff-scheme           = Verlet    ; Buffered neighbor searching
nstlist                 = 10        ; 20 fs neighbor list update

; Electrostatics & VdW (Amber defaults)
rcoulomb                = 1.0       ; short-range electrostatic cutoff (nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (nm)
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT

; Temperature coupling is on
tcoupl                  = V-rescale             ; stochastic Bussi thermostat 
tc-grps                 = Protein Non-Protein   ; Two coupling groups
tau_t                   = 0.1     0.1           ; time constant (ps)
ref_t                   = 298     298           ; reference temperature (K) 

; Pressure coupling is on
pcoupl                  = C-rescale             ; Modern, stable barostat
pcoupltype              = isotropic             ; uniform scaling of box vectors
tau_p                   = 5.0                   ; time constant (ps)
ref_p                   = 1.0                   ; reference pressure (bar)
compressibility         = 4.5e-5                ; isothermal compressibility of water (bar^-1)
refcoord_scaling        = com

; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC

; Velocity generation
gen_vel                 = no        ; Velocity generation is off (read from checkpoint)
"""

with open('/kaggle/working/npt.mdp', 'w') as f:
    f.write(npt_content)
print("npt.mdp successfully created.")

In [ ]:
%%bash

# Target the micromamba GROMACS environment binary
GMX_ENV="micromamba run -n gmx2024_cuda gmx"

echo "=== Running grompp to assemble npt.tpr ==="
$GMX_ENV grompp \
  -f /kaggle/working/npt.mdp \
  -c /kaggle/working/nvt.gro \
  -r /kaggle/working/nvt.gro \
  -t /kaggle/working/nvt.cpt \
  -p /kaggle/working/topol.top \
  -o /kaggle/working/npt.tpr

echo ""
echo "=== Running mdrun NPT Equilibration on GPU ==="
echo "Simulating 500 ps. Progress updates will show below..."
$GMX_ENV mdrun \
  -v \
  -deffnm /kaggle/working/npt \
  -nb gpu \
  -pme gpu

In [ ]:
# Create the md.mdp file tailored for your Amber99sb complex setup with 1 ns checkpoints
md_content = """; md.mdp - Amber99sb-ildn optimized Production MD (10 ns)
title                   = Amber Complex 10ns Production MD 

; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = 5000000   ; 2 fs * 5,000,000 = 10,000 ps (10 ns)
dt                      = 0.002     ; 2 fs

; Output control (500,000 steps = 1000 ps = 1 ns)
nstxout                 = 0         ; suppress bulky uncompressed .trr coordinates
nstvout                 = 0         ; suppress bulky uncompressed .trr velocities
nstfout                 = 0         ; suppress bulky uncompressed .trr forces
nstenergy               = 5000      ; save energies every 10.0 ps
nstlog                  = 5000      ; update log file every 10.0 ps
nstxout-compressed      = 5000      ; save compressed trajectory (.xtc) every 10.0 ps
compressed-x-grps       = System    ; save the whole system

; Bond parameters
continuation            = yes       ; Restarting after NPT
constraint_algorithm    = lincs     ; holonomic constraints 
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy

; Nonbonded settings 
cutoff-scheme           = Verlet    ; Buffered neighbor searching
nstlist                 = 10        ; 20 fs neighbor list update

; Electrostatics & VdW (Amber defaults)
rcoulomb                = 1.0       ; short-range electrostatic cutoff (nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (nm)
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT

; Temperature coupling is on
tcoupl                  = V-rescale             ; stochastic Bussi thermostat 
tc-grps                 = Protein Non-Protein   ; Two coupling groups
tau_t                   = 0.1     0.1           ; time constant (ps)
ref_t                   = 298     298           ; reference temperature (K) 

; Pressure coupling is on
pcoupl                  = C-rescale             ; Modern, stable barostat
pcoupltype              = isotropic             ; uniform scaling of box vectors
tau_p                   = 5.0                   ; time constant (ps)
ref_p                   = 1.0                   ; reference pressure (bar)
compressibility         = 4.5e-5                ; isothermal compressibility of water (bar^-1)

; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC

; Velocity generation
gen_vel                 = no        ; Velocity generation is off (read from checkpoint)
"""

with open('/kaggle/working/md.mdp', 'w') as f:
    f.write(md_content)
print("md.mdp successfully created.")

In [ ]:
%%bash

# gromacs binary path for mamba environment
GMX_ENV="micromamba run -n gmx2024_cuda gmx"

echo "=== Running grompp to assemble md_0_10.tpr ==="
$GMX_ENV grompp \
  -f /kaggle/working/md.mdp \
  -c /kaggle/working/npt.gro \
  -t /kaggle/working/npt.cpt \
  -p /kaggle/working/topol.top \
  -o /kaggle/working/md_0_10.tpr

In [ ]:
import subprocess
import sys
import time

print("=== Running 10 ns Production MD with Live Log Updates ===")
print("its starting(buckle up):\n")

# mamba environment and command flags
cmd = [
    "micromamba", "run", "-n", "gmx2024_cuda", "gmx", "mdrun",
    "-v",
    "-deffnm", "/kaggle/working/md_0_10",
    "-cpo", "/kaggle/working/md_0_10.cpt",
    "-cpt", "15",
    "-ntomp", "4",
    "-pin", "on",
    "-nb", "gpu",
    "-pme", "gpu"
]


process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)


try:
    for line in process.stdout:
        if "step" in line or "remaining" in line or "Starting" in line or "Performance" in line:
            sys.stdout.write(f"\r[LIVE LOG] {line.strip()}")
            sys.stdout.flush()
        elif line.strip():
            print(line.strip())
except KeyboardInterrupt:
    print("\n[WARNING] you have stopped the run")
    process.terminate()

process.wait()
print("\n=== MD Simulation Finished Successfully! ===")

In [ ]:
%%bash
GMX_ENV="micromamba run -n gmx2024_cuda gmx"

echo "=== ১. প্রথমে PBC ইফেক্ট ঠিক করে নো-পিবিসি ট্র্যাজেক্টরি তৈরি করছি ==="
# '1' (Protein) সেন্টারিং এর জন্য এবং '0' (System) আউটপুট এর জন্য পাঠানো হচ্ছে
echo "1 0" | $GMX_ENV trjconv \
  -s /kaggle/working/md_0_10.tpr \
  -f /kaggle/working/md_0_10.xtc \
  -o /kaggle/working/md_0_10_noPBC.xtc \
  -pbc mol -center

echo ""
echo "=== ২. এবার Equilibrated Reference-এর সাপেক্ষে ১ম RMSD বের করছি ==="
# Backbone vs Backbone ("4 4")
echo "4 4" | $GMX_ENV rms \
  -s /kaggle/working/md_0_10.tpr \
  -f /kaggle/working/md_0_10_noPBC.xtc \
  -o /kaggle/working/rmsd.xvg \
  -tu ns

echo ""
echo "=== ৩. সবশেষে Crystal (em.tpr) Reference-এর সাপেক্ষে ২য় RMSD বের করছি ==="
# Backbone vs Backbone ("4 4")
echo "4 4" | $GMX_ENV rms \
  -s /kaggle/working/em.tpr \
  -f /kaggle/working/md_0_10_noPBC.xtc \
  -o /kaggle/working/rmsd_xtal.xvg \
  -tu ns

echo ""
echo "=== অল ডান! সব ডেটা ফাইল তৈরি শেষ। ==="

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def read_xvg(filepath):
    time, rmsd = [], []
    with open(filepath, 'r') as f:
        for line in f:
            # কমেন্ট এবং মেটাডেটা বাদ দেওয়া
            if not line.startswith(('@', '#')):
                cols = line.split()
                if len(cols) == 2:
                    time.append(float(cols[0]))
                    rmsd.append(float(cols[1]))
    return np.array(time), np.array(rmsd)

try:
    # ১. ডেটা লোড করা
    time1, rmsd_equil = read_xvg('/kaggle/working/rmsd.xvg')
    time2, rmsd_xtal = read_xvg('/kaggle/working/rmsd_xtal.xvg')

    # ২. স্ট্যাটিস্টিক্যাল ক্যালকুলেশন (পুরো ১০ এনএস এবং শেষ ৫ এনএস এর জন্য)
    mean_equil = np.mean(rmsd_equil)
    std_equil = np.std(rmsd_equil)
    mean_xtal = np.mean(rmsd_xtal)
    std_xtal = np.std(rmsd_xtal)
    
    # শেষ অর্ধেক ট্র্যাজেক্টরি (প্লাটো রিজিয়ন) চেক করার জন্য
    half_idx = len(rmsd_equil) // 2
    stable_mean = np.mean(rmsd_equil[half_idx:])
    stable_std = np.std(rmsd_equil[half_idx:])

    # ৩. ৫০০ পিকোসেকেন্ডের (৫০ ফ্রেম) রানিং অ্যাভারেজ
    df_equil = pd.Series(rmsd_equil).rolling(window=50, min_periods=1).mean()
    df_xtal = pd.Series(rmsd_xtal).rolling(window=50, min_periods=1).mean()

    # ৪. টার্মিনালে প্রফেশনাল সামারি প্রিন্ট করা
    print("="*60)
    print("         GROMACS RMSD SIMULATION ANALYSIS SUMMARY")
    print("="*60)
    print(f"Simulation Time Length  : {time1[-1]:.2f} ns")
    print(f"Total Data Points       : {len(time1)}")
    print("-"*60)
    print(f"RMSD vs Equilibrated Structure (Overall):")
    print(f"  - Mean RMSD           : {mean_equil:.4f} nm ({mean_equil*10:.2f} Å)")
    print(f"  - Std Deviation (SD)  : {std_equil:.4f} nm")
    print(f"  - Max RMSD Spike      : {np.max(rmsd_equil):.4f} nm")
    print("-"*60)
    print(f"RMSD vs Crystal/PDB Structure (Overall):")
    print(f"  - Mean RMSD           : {mean_xtal:.4f} nm ({mean_xtal*10:.2f} Å)")
    print(f"  - Std Deviation (SD)  : {std_xtal:.4f} nm")
    print("-"*60)
    print(f"Stability & Convergence Check (Last 5 ns):")
    print(f"  - Plateau Mean RMSD   : {stable_mean:.4f} nm ± {stable_std:.4f} nm")
    
    if stable_std < 0.02:
        print("  - System Status       : HIGHLY STABLE (Excellent convergence)")
    elif stable_std < 0.05:
        print("  - System Status       : STABLE (Normal thermal fluctuations)")
    else:
        print("  - System Status       : FLUCTUATING (Check for large domain movements)")
    print("="*60)

    # ৫. প্লট জেনারেট করা
    plt.figure(figsize=(10, 5.5))
    
    # Equilibrated Reference (Raw + Smoothed)
    plt.plot(time1, rmsd_equil, color='lightgray', alpha=0.6, label='RMSD vs Equil (Raw)')
    plt.plot(time1, df_equil, color='darkblue', linewidth=2, label='RMSD vs Equil (Running Avg)')
    
    # Crystal Reference (Raw + Smoothed)
    plt.plot(time2, rmsd_xtal, color='pink', alpha=0.5, label='RMSD vs Crystal (Raw)')
    plt.plot(time2, df_xtal, color='crimson', linewidth=2, label='RMSD vs Crystal (Running Avg)')
    
    # লেআউট এবং লেবেল
    plt.title('RMSD Evolution: Equilibrated vs Crystal Structure References', fontsize=12, fontweight='bold')
    plt.xlabel('Time (ns)', fontsize=11)
    plt.ylabel('RMSD (nm)', fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    
    # ইমেজ ডিস্কে সেভ করা
    plt.savefig('/kaggle/working/rmsd_final_report.png', dpi=300)
    plt.show()

except FileNotFoundError:
    print("[ERROR] rmsd.xvg বা rmsd_xtal.xvg ফাইলটি পাওয়া যায়নি। দয়া করে নিশ্চিত করুন আগের সেলের গ্রোম্যাক্স রানটি ঠিকঠাক শেষ হয়েছে কিনা।")

In [ ]:
%%bash
GMX_ENV="micromamba run -n gmx2024_cuda gmx"

# -sel Protein ফ্লাগ দিয়ে সরাসরি প্রোটিনের Rg বের করা হচ্ছে
$GMX_ENV gyrate \
  -s /kaggle/working/md_0_10.tpr \
  -f /kaggle/working/md_0_10_noPBC.xtc \
  -o /kaggle/working/gyrate.xvg \
  -sel Protein \
  -tu ns

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def read_gyrate(filepath):
    time, rg = [], []
    with open(filepath, 'r') as f:
        for line in f:
            if not line.startswith(('@', '#')):
                cols = line.split()
                if len(cols) >= 2:
                    time.append(float(cols[0]))
                    rg.append(float(cols[1])) # Total Rg সাধারণত ২য় কলামে থাকে
    return np.array(time), np.array(rg)

time, rg = read_gyrate('/kaggle/working/gyrate.xvg')
mean_rg = np.mean(rg)
std_rg = np.std(rg)

print("="*60)
print("         RADIUS OF GYRATION (Rg) ANALYSIS SUMMARY")
print("="*60)
print(f"Mean Rg Value         : {mean_rg:.4f} nm ({mean_rg*10:.2f} Å)")
print(f"Standard Deviation (SD): {std_rg:.4f} nm")
if std_rg < 0.015:
    print("System Compactness    : HIGHLY COMPACT & FOLDED (No unfolding detected)")
else:
    print("System Compactness    : FLUCTUATING (Possible structural expansion)")
print("="*60)

# ৫00 ps রানিং অ্যাভারেজ
df_rg = pd.Series(rg).rolling(window=50, min_periods=1).mean()

plt.figure(figsize=(9, 5))
plt.plot(time, rg, color='lightgreen', alpha=0.6, label='Rg (Raw Data)')
plt.plot(time, df_rg, color='darkgreen', linewidth=2, label='Rg (500-ps Running Avg)')
plt.title('Radius of Gyration: Structural Compactness Over Time', fontsize=12, fontweight='bold')
plt.xlabel('Time (ns)', fontsize=11)
plt.ylabel('Radius of Gyration Rg (nm)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
%%bash
GMX_ENV="micromamba run -n gmx2024_cuda gmx"
TPR="/kaggle/working/md_0_10.tpr"
XTC="/kaggle/working/md_0_10_noPBC.xtc"
NDX="/kaggle/working/index.ndx"

# =============================================
# ১. ইনডেক্স ফাইল তৈরি (নন-ইন্টার‍্যাক্টিভ)
# =============================================
# 'q' পাঠিয়ে সাথে সাথে save করে বেরিয়ে আসা
echo "q" | $GMX_ENV make_ndx -f $TPR -o $NDX 2>&1 | tail -5

echo ""
echo "=== ইনডেক্স ফাইলের গ্রুপ চেক ==="
$GMX_ENV dump -s $TPR 2>&1 | grep -E "group|Protein|SOL|MainChain|SideChain" | head -20

echo ""
echo "=== ১. মেইনচেইন-মেইনচেইন H-বন্ড ==="
# MainChain+H (গ্রুপ 7) → MainChain+H (গ্রুপ 7)
printf "7\n7\n" | $GMX_ENV hbond \
  -s $TPR \
  -f $XTC \
  -n $NDX \
  -num /kaggle/working/hbnum_mainchain.xvg \
  -tu ns 2>&1 | tail -10

echo ""
echo "=== ২. সাইডচেইন-সাইডচেইন H-বন্ড ==="
# SideChain (গ্রুপ 8) → SideChain (গ্রুপ 8)
printf "8\n8\n" | $GMX_ENV hbond \
  -s $TPR \
  -f $XTC \
  -n $NDX \
  -num /kaggle/working/hbnum_sidechain.xvg \
  -tu ns 2>&1 | tail -10

echo ""
echo "=== ৩. প্রোটিন → পানি H-বন্ড ==="
# Protein (গ্রুপ 1) → SOL (গ্রুপ 13)
printf "1\n13\n" | $GMX_ENV hbond \
  -s $TPR \
  -f $XTC \
  -n $NDX \
  -num /kaggle/working/hbnum_prot_to_sol.xvg \
  -tu ns 2>&1 | tail -10

echo ""
echo "=== ৪. পানি → প্রোটিন H-বন্ড (উল্টো) ==="
# SOL (গ্রুপ 13) → Protein (গ্রুপ 1)
printf "13\n1\n" | $GMX_ENV hbond \
  -s $TPR \
  -f $XTC \
  -n $NDX \
  -num /kaggle/working/hbnum_sol_to_prot.xvg \
  -tu ns 2>&1 | tail -10

echo ""
echo "=== সব কাজ শেষ! ফাইল চেক ==="
ls -lh /kaggle/working/*.xvg

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# =============================================
# হাইড্রোজেন বন্ড .xvg ফাইল পড়ার ফাংশন
# =============================================
def read_hbond(filepath):
    time, hb = [], []
    with open(filepath, 'r') as f:
        for line in f:
            if not line.startswith(('@', '#')):
                cols = line.split()
                if len(cols) >= 2:
                    time.append(float(cols[0]))
                    hb.append(int(cols[1]))
    return np.array(time), np.array(hb)

# =============================================
# ডাটা লোড
# =============================================
t1, hb_main = read_hbond('/kaggle/working/hbnum_mainchain.xvg')
t2, hb_side = read_hbond('/kaggle/working/hbnum_sidechain.xvg')
t3, hb_prot_to_sol = read_hbond('/kaggle/working/hbnum_prot_to_sol.xvg')
t4, hb_sol_to_prot = read_hbond('/kaggle/working/hbnum_sol_to_prot.xvg')

# মোট প্রোটিন-পানি H-বন্ড (উভয় দিক)
hb_wat_total = hb_prot_to_sol + hb_sol_to_prot

# =============================================
# পরিসংখ্যান
# =============================================
print("=" * 65)
print("         HYDROGEN BOND (H-BOND) ANALYSIS SUMMARY")
print("=" * 65)
print(f"Protein Backbone H-Bonds (Avg)     : {np.mean(hb_main):.1f} ± {np.std(hb_main):.1f}")
print(f"Protein Sidechain H-Bonds (Avg)    : {np.mean(hb_side):.1f} ± {np.std(hb_side):.1f}")
print(f"Protein → Water H-Bonds (Avg)      : {np.mean(hb_prot_to_sol):.1f} ± {np.std(hb_prot_to_sol):.1f}")
print(f"Water → Protein H-Bonds (Avg)      : {np.mean(hb_sol_to_prot):.1f} ± {np.std(hb_sol_to_prot):.1f}")
print(f"Total Protein-Water H-Bonds (Avg)  : {np.mean(hb_wat_total):.1f} ± {np.std(hb_wat_total):.1f}")


# =============================================
# ৫০ ফ্রেম রানিং অ্যাভারেজ (মসৃণ করার জন্য)
# =============================================
window = 50
df_main = pd.Series(hb_main).rolling(window=window, min_periods=1).mean()
df_side = pd.Series(hb_side).rolling(window=window, min_periods=1).mean()
df_wat  = pd.Series(hb_wat_total).rolling(window=window, min_periods=1).mean()

# =============================================
# প্লট তৈরি
# =============================================
plt.figure(figsize=(12, 7))

plt.plot(t3, df_wat, color='blue', linewidth=2, label='Total Protein-Water (SOL)')
plt.plot(t1, df_main, color='red', linewidth=2, label='Backbone (MainChain+H)')
plt.plot(t2, df_side, color='orange', linewidth=2, label='Sidechain-Sidechain')

plt.title('Hydrogen Bonds Evolution Profile (10 ns MD Simulation)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Time (ns)', fontsize=12)
plt.ylabel('Number of Hydrogen Bonds', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='best', fontsize=10, framealpha=0.9)

# গড় লাইন যোগ করা
plt.axhline(y=np.mean(hb_main), color='red', linestyle=':', alpha=0.5)
plt.axhline(y=np.mean(hb_side), color='orange', linestyle=':', alpha=0.5)
plt.axhline(y=np.mean(hb_wat_total), color='blue', linestyle=':', alpha=0.5)

plt.tight_layout()

# =============================================
# প্লট সেভ (উচ্চ রেজোলিউশন)
# =============================================
output_path = '/kaggle/working/hbond_analysis_plot.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✅ প্লট সেভ হয়েছে: {output_path}")

# স্ক্রিনেও দেখানো
plt.show()

# =============================================
# অতিরিক্ত: CSV ফাইলে ডাটা সেভ
# =============================================
df_export = pd.DataFrame({
    'Time_ns': t1,
    'Backbone_HB': hb_main,
    'Sidechain_HB': hb_side,
    'Prot_to_Sol_HB': hb_prot_to_sol,
    'Sol_to_Prot_HB': hb_sol_to_prot,
    'Total_Prot_Water_HB': hb_wat_total
})
csv_path = '/kaggle/working/hbond_data_summary.csv'
df_export.to_csv(csv_path, index=False)
print(f"✅ CSV ডাটা সেভ হয়েছে: {csv_path}")

# =============================================
# বক্সপ্লট কম্প্যারিজন (অপশনাল)
# =============================================
fig, ax = plt.subplots(figsize=(8, 5))
box_data = [hb_main, hb_side, hb_wat_total]
labels = ['Backbone', 'Sidechain', 'Total\nProtein-Water']
bp = ax.boxplot(box_data, labels=labels, patch_artist=True, widths=0.5)

colors = ['#FF6B6B', '#FFA726', '#42A5F5']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Number of Hydrogen Bonds', fontsize=12)
ax.set_title('H-Bond Distribution Comparison', fontsize=13, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.3, axis='y')

box_path = '/kaggle/working/hbond_boxplot.png'
plt.tight_layout()
plt.savefig(box_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ বক্সপ্লট সেভ হয়েছে: {box_path}")
plt.show()

In [ ]:
%%bash
GMX_ENV="micromamba run -n gmx2024_cuda gmx"

$GMX_ENV dssp \
  -s /kaggle/working/md_0_10.tpr \
  -f /kaggle/working/md_0_10_noPBC.xtc \
  -tu ns \
  -o /kaggle/working/dssp.dat \
  -num /kaggle/working/dssp_num.xvg

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import pandas as pd
import re
import os

# =============================================
# ১. ফাইল চেক ও ডাটা লোড (সেফগার্ড সহ)
# =============================================
dat_path = '/kaggle/working/dssp.dat'
num_path = '/kaggle/working/dssp_num.xvg'

# --- .dat ফাইল পড়া ---
def read_dssp_dat(filepath):
    ss_data = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            # =, P, ~, space-সহ সব DSSP কোড ধরার জন্য রেজেক্স
            if re.match(r'^[HEBGITS~P= ]+$', line):
                ss_data.append(line)
    return np.array(ss_data)

print("=" * 60)
print("  DSSP DATA LOADING STATUS")
print("=" * 60)
if not os.path.exists(dat_path):
    raise FileNotFoundError(f"{dat_path} পাওয়া যায়নি!")
ss_data = read_dssp_dat(dat_path)
n_frames = len(ss_data)
if n_frames == 0:
    raise ValueError("DSSP .dat ফাইলে কোনো স্ট্রাকচার লাইন পাওয়া যায়নি।")
n_residues = len(ss_data[0])
print(f".dat ফাইল থেকে {n_frames} টি ফ্রেম পাওয়া গেল, প্রতি ফ্রেমে {n_residues} টি রেসিডিউ।")

# --- .xvg ফাইল থেকে সময় পড়া (যদি থাকে) ---
time_arr = None
if os.path.exists(num_path) and os.path.getsize(num_path) > 0:
    time_list = []
    ss_counts_list = []
    with open(num_path, 'r') as f:
        for line in f:
            if not line.startswith(('@', '#')):
                cols = line.split()
                if len(cols) >= 2:
                    time_list.append(float(cols[0]))
                    ss_counts_list.append([float(x) for x in cols[1:]])
    if len(time_list) == n_frames:
        time_arr = np.array(time_list)
        ss_counts_arr = np.array(ss_counts_list)
        print(f".xvg ফাইল থেকেও সময় ({len(time_arr)} ফ্রেম) সফলভাবে লোড হয়েছে।")
    else:
        print(f".xvg ফাইলে ফ্রেম সংখ্যা ({len(time_list)}) .dat ফাইলের সাথে মেলেনি। তাই সময় গণনা করা হবে।")

# --- ফ্রেম সংখ্যা থেকে সময় তৈরি (ফলব্যাক) ---
if time_arr is None:
    # ১০ ns সিমুলেশন, ১০০১ ফ্রেম (০ থেকে ১০০০ inclusive) -> dt = 10/1000 = 0.01 ns
    dt = 10.0 / (n_frames - 1)  # 0.01 ns
    time_arr = np.arange(n_frames) * dt
    # .xvg ছাড়া ss_counts_arr দরকার নেই, আমরা .dat থেকেই গুণব
    ss_counts_arr = None
    print(f".xvg অনুপস্থিত/ত্রুটিপূর্ণ। সময় অ্যারে তৈরি: {time_arr[0]:.2f} - {time_arr[-1]:.2f} ns (dt={dt:.3f} ns)")

print(f"সময়সীমা: {time_arr[0]:.2f} - {time_arr[-1]:.2f} ns\n")

# =============================================
# ২. সেকেন্ডারি স্ট্রাকচার কালার ও লেবেল
# =============================================
ss_color_dict = {
    'H': '#FF0000',  # Alpha-helix
    'B': '#000000',  # Beta-bridge
    'E': '#FFFF00',  # Beta-strand
    'G': '#FF8C00',  # 3-10 helix
    'I': '#8B00FF',  # Pi-helix
    'T': '#008B8B',  # Turn
    'S': '#00CED1',  # Bend
    'P': '#FFC0CB',  # PPII
    '~': '#D3D3D3',  # Loop
    ' ': '#FFFFFF',  # Coil
    '=': '#E6E6FA',  # Other (ল্যাভেন্ডার)
}

ss_labels_dict = {
    'H': 'α-Helix', 'B': 'β-Bridge', 'E': 'β-Strand',
    'G': '3₁₀-Helix', 'I': 'π-Helix', 'T': 'Turn',
    'S': 'Bend', 'P': 'PPII', '~': 'Loop', ' ': 'Coil',
    '=': 'Other'
}

# =============================================
# ৩. স্ট্রাকচার কাউন্টিং (.dat থেকে সরাসরি)
# =============================================
ss_counts_dict = {ch: np.zeros(n_frames) for ch in ss_color_dict if ch != ' '}
coil_count = np.zeros(n_frames)

for i, ss_str in enumerate(ss_data):
    for ch in ss_str:
        if ch in ss_counts_dict:
            ss_counts_dict[ch][i] += 1
        elif ch == ' ':
            coil_count[i] += 1

# =============================================
# ৪. গড় স্ট্রাকচার কন্টেন্ট প্রিন্ট
# =============================================
print("গড় সেকেন্ডারি স্ট্রাকচার কন্টেন্ট:")
print("-" * 50)
for ch in ['H', 'E', 'B', 'T', 'S', 'G', 'I', 'P', '~', '=', ' ']:
    if ch in ss_counts_dict:
        avg = np.mean(ss_counts_dict[ch])
        pct = avg / n_residues * 100
        print(f"  {ss_labels_dict[ch]:15s} ({ch}): {avg:6.1f} ({pct:5.1f}%)")
    elif ch == ' ':
        avg = np.mean(coil_count)
        pct = avg / n_residues * 100
        print(f"  {ss_labels_dict[ch]:15s} ({ch}): {avg:6.1f} ({pct:5.1f}%)")
print("-" * 50)

# =============================================
# ৫. প্লট ১: DSSP 2D টাইমলাইন + কন্টেন্ট
# =============================================
unique_chars = sorted(set(''.join(ss_data)))
char_to_num = {c: i+1 for i, c in enumerate(unique_chars)}

ss_matrix = np.zeros((n_frames, n_residues))
for i, ss_str in enumerate(ss_data):
    for j, ch in enumerate(ss_str):
        if j < n_residues:
            ss_matrix[i, j] = char_to_num.get(ch, 0)

# কালারম্যাপ: আসল ক্যারেক্টার অর্ডারে
color_order = [' ', '~', '=', 'S', 'T', 'P', 'I', 'G', 'B', 'E', 'H']
cmap_colors = [ss_color_dict[c] for c in color_order if c in unique_chars]

cmap = ListedColormap(cmap_colors)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
im = ax1.imshow(ss_matrix.T, aspect='auto', origin='lower',
                extent=[time_arr[0], time_arr[-1], 1, n_residues],
                cmap=cmap, vmin=0.5, vmax=len(cmap_colors)+0.5,
                interpolation='none')
ax1.set_xlabel('Time (ns)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Residue Number', fontsize=12, fontweight='bold')
ax1.set_title(f'Secondary Structure Evolution (DSSP)\n{n_residues} residues, {n_frames} frames',
              fontsize=14, fontweight='bold')

legend_patches = []
for c in color_order:
    if c in unique_chars:
        legend_patches.append(mpatches.Patch(color=ss_color_dict[c],
                                            label=f'{c} = {ss_labels_dict[c]}'))
ax1.legend(handles=legend_patches, bbox_to_anchor=(1.02, 1),
           loc='upper left', fontsize=8, framealpha=0.9, ncol=2)
ax1.grid(False)

# টাইম-কোর্স
ax2.plot(time_arr, ss_counts_dict['H'], 'red', linewidth=2.5, label='α-Helix (H)')
ax2.plot(time_arr, ss_counts_dict['E'], 'gold', linewidth=2, label='β-Strand (E)')
ax2.plot(time_arr, ss_counts_dict['B'], 'black', linewidth=1.5, alpha=0.7, label='β-Bridge (B)')
ax2.plot(time_arr, ss_counts_dict['T'], 'teal', linewidth=1.5, alpha=0.7, label='Turn (T)')
ax2.plot(time_arr, ss_counts_dict['S'], 'cyan', linewidth=1, alpha=0.6, label='Bend (S)')
if 'G' in ss_counts_dict:
    ax2.plot(time_arr, ss_counts_dict['G'], 'orange', linewidth=1, alpha=0.6, label='3₁₀-Helix (G)')
if 'I' in ss_counts_dict:
    ax2.plot(time_arr, ss_counts_dict['I'], 'purple', linewidth=1, alpha=0.6, label='π-Helix (I)')
ax2.plot(time_arr, coil_count, 'gray', linewidth=1.5, alpha=0.7, linestyle='--', label='Coil')
ax2.set_xlabel('Time (ns)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Residues', fontsize=12, fontweight='bold')
ax2.set_title('Secondary Structure Content Over Time', fontsize=14, fontweight='bold')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, framealpha=0.9)
ax2.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plot1 = '/kaggle/working/dssp_timeline.png'
plt.savefig(plot1, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ প্লট ১ সেভ: {plot1}")
plt.show()

# =============================================
# ৬. বার চার্ট (গড় কন্টেন্ট)
# =============================================
fig, ax = plt.subplots(figsize=(12, 6))
plot_order = ['H', 'E', 'B', 'T', 'S', 'G', 'I', 'P', '~', '=', ' ']
bar_labels, bar_vals, bar_cols, bar_errs = [], [], [], []
for ch in plot_order:
    if ch in ss_counts_dict:
        val = np.mean(ss_counts_dict[ch])
        if val > 0.05:
            bar_labels.append(f'{ss_labels_dict[ch]}\n({ch})')
            bar_vals.append(val)
            bar_cols.append(ss_color_dict[ch])
            bar_errs.append(np.std(ss_counts_dict[ch]))
    elif ch == ' ':
        val = np.mean(coil_count)
        if val > 0.05:
            bar_labels.append(f'{ss_labels_dict[ch]}\n( )')
            bar_vals.append(val)
            bar_cols.append(ss_color_dict[ch])
            bar_errs.append(np.std(coil_count))

x = np.arange(len(bar_labels))
bars = ax.bar(x, bar_vals, yerr=bar_errs, color=bar_cols, alpha=0.85,
              capsize=5, edgecolor='black', linewidth=0.8)
for bar, v in zip(bars, bar_vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height()+0.5,
            f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(bar_labels, fontsize=9)
ax.set_ylabel('Average Number of Residues', fontsize=12, fontweight='bold')
ax.set_title(f'Average Secondary Structure Content\n({n_residues} residues, 10 ns)', fontsize=14, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
bar_path = '/kaggle/working/dssp_barplot.png'
plt.savefig(bar_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ প্লট ২ সেভ: {bar_path}")
plt.show()

# =============================================
# ৭. স্ট্যাকড এরিয়া (কম্পোজিশন %)
# =============================================
fig, ax = plt.subplots(figsize=(12, 5))
helix = ss_counts_dict['H'] + ss_counts_dict.get('G', 0) + ss_counts_dict.get('I', 0)
sheet = ss_counts_dict['E'] + ss_counts_dict['B']
turn_bend = ss_counts_dict['T'] + ss_counts_dict['S']
coil = coil_count
other = ss_counts_dict.get('P', 0) + ss_counts_dict.get('~', 0) + ss_counts_dict.get('=', 0)

total = n_residues
ax.fill_between(time_arr, 0, helix/total*100, color='red', alpha=0.8, label='Helix')
ax.fill_between(time_arr, helix/total*100, (helix+sheet)/total*100, color='gold', alpha=0.8, label='β-Sheet')
ax.fill_between(time_arr, (helix+sheet)/total*100, (helix+sheet+turn_bend)/total*100, color='teal', alpha=0.8, label='Turn/Bend')
ax.fill_between(time_arr, (helix+sheet+turn_bend)/total*100, (helix+sheet+turn_bend+coil)/total*100, color='gray', alpha=0.8, label='Coil')
ax.fill_between(time_arr, (helix+sheet+turn_bend+coil)/total*100, 100, color='lightgray', alpha=0.8, label='Other')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Percentage (%)')
ax.set_title('Secondary Structure Composition')
ax.set_ylim(0, 100)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
comp_path = '/kaggle/working/dssp_composition.png'
plt.savefig(comp_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ প্লট ৩ সেভ: {comp_path}")
plt.show()

# =============================================
# ৮. CSV এক্সপোর্ট
# =============================================
csv_dict = {'Time_ns': time_arr}
for ch in ['H','E','B','T','S','G','I','P','~','=']:
    if ch in ss_counts_dict:
        csv_dict[f'{ss_labels_dict[ch]}_{ch}'] = ss_counts_dict[ch]
csv_dict['Coil'] = coil_count
df = pd.DataFrame(csv_dict)
df.to_csv('/kaggle/working/dssp_summary.csv', index=False)
print(f"✅ CSV সেভ: /kaggle/working/dssp_summary.csv")
print("\n🎉 সম্পূর্ণ DSSP বিশ্লেষণ শেষ!")

In [ ]:
!ls -lh /kaggle/working/

In [ ]:
!rm -f /kaggle/working/#*#
!rm -f /kaggle/working/md_0_10.xtc
!rm -f /kaggle/working/*.trr

In [ ]:
import os
import zipfile

# জিপ ফাইলের নাম এবং যে ফাইলগুলো আমরা নেব
zip_name = "/kaggle/working/GROMACS_10ns_Analysis_Backup.zip"

# আবশ্যিক এক্সটেনশন যা আমরা ব্যাকআপ রাখতে চাই
allowed_extensions = ['.png', '.xvg', '.csv', '.top', '.itp', '.mdp', '.cpt', '.tpr']
# নির্দিষ্ট বড় ফাইল যা অবশ্যই লাগবে
important_large_files = ['md_0_10_noPBC.xtc', 'md_0_10.gro', 'em.tpr']

print("=== জিপিং প্রসেস শুরু হচ্ছে... ===")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk("/kaggle/working/"):
        # ইন্সটলার ফোল্ডার এবং ব্যাকআপ ফাইল (#...#) স্কিপ করা
        if "Gromacs_2024.4_stable_installer" in root:
            continue
            
        for file in files:
            if file.startswith('#') or file.endswith('#'):
                continue
                
            file_path = os.path.join(root, file)
            rel_path = os.path.relpath(file_path, "/kaggle/working/")
            
            # ফিল্টারিং লজিক
            _, ext = os.path.splitext(file)
            if ext in allowed_extensions or file in important_large_files:
                print(f"Adding to ZIP: {rel_path}")
                zipf.write(file_path, rel_path)

print(f"\n✅ সফলভাবে জিপ ফাইল তৈরি হয়েছে: {zip_name}")
print(f"সাইজ: {os.path.getsize(zip_name) / (1024*1024):.2f} MB")